In [ ]:
%matplotlib inline
from pathlib import Path
import sys
import shutil
import math
import os
import json
import random
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm, trange
import torchmetrics
import matplotlib.pyplot as plt
from pprint import pprint
import utils as course_utils

# ======== Your controls: change these to run the notebook your way ========
# True: a small run on a tiny generated dataset; grading always uses it.
# False: training on real MNIST (downloaded to ~/ once, about 60 MB).
IS_UNITTEST = True
# Epochs per training run when IS_UNITTEST = False (the small run trains 1).
FULL_RUN_EPOCHS = 3
# ==========================================================================
# Grading always uses the small run, whatever IS_UNITTEST is set to above.
IS_UNITTEST = IS_UNITTEST or course_utils.checker.is_unittest_run()
# Thread count changes the order of float sums; the small run must
# repeat the teacher's numbers on any machine.
if IS_UNITTEST:
    torch.set_num_threads(1)

# -- start --
data_dir = course_utils.checker.get_data_dir()
course_utils.checker.save_npz({'seed': np.array(0)}, data_dir / 'inputs_public.npz')
course_utils.checker.save_npz({'seed': np.array(42)}, data_dir / 'inputs_private.npz')
if not course_utils.checker.is_valid_npz('xinputs.npz'):
    shutil.copyfile(data_dir / 'inputs_public.npz', 'xinputs.npz')
# -- end --

seed = int(course_utils.checker.load_answer('xinputs.npz')['seed'])

import source
# -- start --
# The teacher's notebook runs with the teacher's copy of the shared code.
if course_utils.hidden.detect():
    course_utils.hidden.replace_code()
# -- end --
src = source.code.src
answer = {}


# Simple GAN With Gradient Reversal

A generative adversarial network has two trainable parts:

- a generator that maps a random noise vector to an image-shaped vector;
- a discriminator that receives both generated and real images and predicts which inputs are real.

In this exercise we use one shared backpropagation pass instead of alternating separate generator and discriminator steps. The generated samples pass through a `GradientReversalLayer` before the discriminator sees them:

`noise -> generator -> gradient reversal -> discriminator -> classifier`

The gradient-reversal layer does nothing to the forward value. During backpropagation it multiplies the gradient by `-strength`, so the discriminator learns the normal real/fake classification task while the generator receives the opposite signal and learns to make fake images look more real.

The networks use `LeakyReLU(negative_slope=0.5)` by default. The slope is deliberately large: if the discriminator becomes much better than the generator, ordinary ReLU-style activations and saturated classifier logits can leave the generator with tiny useful gradients. A strong leaky branch keeps negative activations informative and helps gradients keep flowing through the discriminator path back toward the generator.


# Task 0: Prepare the Environment

## Goal

Set all random seeds with `src.utils.seed.seed_all(0)`. Reproducibility matters here because generated examples and training curves are easier to compare when initialization and dataloader shuffling are deterministic.

## Implementation Details

Write the seed call in the cell below. The expected student code should be placed after `## YOUR CODE HERE`.


In [ ]:
# Edit this cell in task.ipynb.
## YOUR CODE HERE
# -- start --
src.utils.seed.seed_all(0)
# -- end --

seed_check = {
    'python.random': [random.random() for _ in range(4)],
    'numpy.random.rand': np.random.rand(2, 3),
    'torch.rand': torch.rand(2, 3),
    'torch.backends.cudnn.deterministic': torch.backends.cudnn.deterministic,
}
answer['seed_check'] = seed_check
pprint(seed_check)


In [ ]:
course_utils.checker.print_progress(
    'Task 0: Prepare the Environment', answer,
    prefixes=('seed_check',),
    is_small_run=IS_UNITTEST)


# Task 1: Reuse MNIST and Add Noise

Reuse `MNISTSimpleDataset` from the earlier tasks and wrap it in `NoiseMNISTDataset`. The wrapper should return the real MNIST sample together with a deterministic random vector that will become the generator input.

Implement `NoiseMNISTDataset.__getitem__` so that every item is a dictionary with:

- `noise`: a vector of length `self.noise_dim`, sampled from a standard normal distribution;
- `image`: the real MNIST image from the base dataset;
- `label`: the original MNIST digit label, used only for visualization.

The noise must be deterministic for each index. Create a local `torch.Generator`, seed it with `self.base_seed + int(index)`, and pass it to `torch.randn`. Use the same dtype as `sample['image']`.

After the class is implemented, set:

- `noise_dim = 256`;
- `MNIST_train = src.datasets.mnist_simple.MNISTSimpleDataset(train=True, small=IS_UNITTEST, seed=seed)`;
- `MNIST_valid = src.datasets.mnist_simple.MNISTSimpleDataset(train=False, small=IS_UNITTEST, seed=seed)`;
- `train_dataset = NoiseMNISTDataset(MNIST_train, noise_dim=noise_dim, base_seed=10_000)`;
- `valid_dataset = NoiseMNISTDataset(MNIST_valid, noise_dim=noise_dim, base_seed=20_000)`.


In [ ]:
# Edit this cell in task.ipynb.
class NoiseMNISTDataset(torch.utils.data.Dataset):
    def __init__(self, base_dataset, noise_dim, base_seed=0):
        self.base_dataset = base_dataset
        self.noise_dim = int(noise_dim)
        self.base_seed = int(base_seed)

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, index):
        ## YOUR CODE HERE
        # -- start --
        sample = self.base_dataset[index]
        generator = torch.Generator()
        generator.manual_seed(self.base_seed + int(index))
        noise = torch.randn(self.noise_dim, generator=generator, dtype=sample['image'].dtype)
        return {
            'noise': noise,
            'image': sample['image'],
            'label': sample['label'],
        }
        # -- end --
        sample = self.base_dataset[index]
        return {
            'noise': torch.zeros(self.noise_dim, dtype=sample['image'].dtype),
            'image': sample['image'],
            'label': sample['label'],
        }


noise_dim = 256
MNIST_train = [{'image': torch.zeros(28, 28), 'label': torch.tensor(0)}] * 32
MNIST_valid = [{'image': torch.zeros(28, 28), 'label': torch.tensor(0)}] * 32
train_dataset = NoiseMNISTDataset(MNIST_train, noise_dim=noise_dim)
valid_dataset = NoiseMNISTDataset(MNIST_valid, noise_dim=noise_dim)

## YOUR CODE HERE
# -- start --
MNIST_train = src.datasets.mnist_simple.MNISTSimpleDataset(train=True, small=IS_UNITTEST, seed=seed)
MNIST_valid = src.datasets.mnist_simple.MNISTSimpleDataset(train=False, small=IS_UNITTEST, seed=seed)
train_dataset = NoiseMNISTDataset(MNIST_train, noise_dim=noise_dim, base_seed=10_000)
valid_dataset = NoiseMNISTDataset(MNIST_valid, noise_dim=noise_dim, base_seed=20_000)
# -- end --

train_sample = train_dataset[0]
valid_sample = valid_dataset[0]

print('noise dtype:', train_sample['noise'].dtype)
print('noise shape:', train_sample['noise'].shape)
print('image dtype:', train_sample['image'].dtype)
print('image shape:', train_sample['image'].shape)

fig, axes = plt.subplots(1, 2, figsize=(5, 2.4), constrained_layout=True)
axes[0].plot(train_sample['noise'])
axes[0].set_title('noise')
axes[1].imshow(train_sample['image'], cmap='gray')
axes[1].set_title(f"real digit {int(train_sample['label'])}")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.show()

answer['noise_shape'] = list(train_sample['noise'].shape)
answer['noise_dtype'] = str(train_sample['noise'].dtype)
answer['image_shape'] = list(train_sample['image'].shape)
answer['image_dtype'] = str(train_sample['image'].dtype)
answer['label_dtype'] = str(train_sample['label'].dtype)
answer['noise_mean'] = float(train_sample['noise'].mean())
answer['image_mean'] = float(train_sample['image'].mean())


In [ ]:
course_utils.checker.print_progress(
    'Task 1: Reuse MNIST and Add Noise', answer,
    prefixes=('noise_', 'image_', 'label_'),
    is_small_run=IS_UNITTEST)


# Task 2: Build the Gradient-Reversal GAN

Edit [src/models/feedforward/adversarial.py](../../source/code/src/models/feedforward/adversarial.py) and implement the `GAN` class. The implementation belongs in `adversarial.py`, next to `GradientReversalLayer`.

Implement the two placeholder regions: `__init__` and `forward`.

In `__init__`, create these modules:

- `self.generator_discriminator_bridge = GradientReversalLayer(gradient_reversal_strength)`;
- `self.gradient_reversal` as an alias of the same layer;
- `self.generator`: fully connected layers following `channels` from left to right. For this task the check uses `[noise_dim, 512, 28 * 28]`, so the generator path is `noise_dim -> 512 -> 784`. Use a fresh copy of the activation after each hidden linear layer, then `torch.nn.Tanh()` at the end so generated pixels live near the MNIST `[-1, 1]` scale;
- `self.discriminator`: fully connected layers following `channels` in reverse. For `[noise_dim, 512, 28 * 28]`, the discriminator path is `784 -> 512 -> noise_dim`, again using fresh activation modules between linear layers;
- `self.classifier`: a final `Linear(noise_dim, 1)` that produces one raw real/fake logit per image.

The default activation is `LeakyReLU(negative_slope=0.5)`. Keep that high negative slope unless you are deliberately experimenting: it helps fight low-gradient behavior when the discriminator gets too confident too early.

In `forward`, implement this data path:

`noise -> generator -> gradient reversal -> concatenate with real images -> discriminator -> classifier`

The model reads `batch['data']['noise']` and either `batch['data']['real']` or `batch['data']['image']`. Flatten real images before concatenation. The classifier outputs raw logits: larger logits mean "more real", smaller logits mean "more fake".

`forward` should mutate and return the same batch dictionary. Keep the original `batch['data']` inputs in place, and add two new dictionaries: `batch['signals']` for tensors used by losses/metrics and `batch['postprocessed']` for easy inspection. If the noise batch has size `B`, then `generated` has shape `(B, 784)`. When real images are present, run the discriminator on fake samples first and real samples second, so the combined discriminator logits have shape `(2 * B,)`, `fake_logits` are the first `B` entries, and `real_logits` are the final `B` entries.

Use exactly these key names. The required outputs are:

- `batch['signals']['generated']`;
- `batch['signals']['discriminator_logits']` for the concatenated fake plus real batch;
- `batch['signals']['fake_logits']`;
- `batch['signals']['real_logits']` when real images are present;
- matching convenience aliases `discriminator_scores`, `fake_scores`, and `real_scores`;
- `batch['postprocessed']['discriminator_score']`, `batch['postprocessed']['fake_score']`, and `batch['postprocessed']['real_score']` when real images are present;
- `batch['postprocessed']['discriminator_probability']`, `batch['postprocessed']['fake_probability']`, and `batch['postprocessed']['real_probability']` when real images are present, computed with `torch.sigmoid`.

A separate `discriminate` helper is not part of the required student-facing API. It may be kept as a small internal helper if the surrounding file already has it, but the required behavior is the `forward` protocol above.


In [ ]:
gan_model = src.models.feedforward.adversarial.GAN(
    [noise_dim, 512, 28 * 28],
    gradient_reversal_strength=1.0
)
src.utils.deterministic_init(gan_model)

check_batch = {
    'data': {
        'noise': torch.randn(10, noise_dim),
        'image': torch.randn(10, 28, 28),
    }
}
with torch.no_grad():
    gan_model(check_batch)

required_signal_keys = {
    'generated',
    'discriminator_logits',
    'fake_logits',
    'real_logits',
    'discriminator_scores',
    'fake_scores',
    'real_scores',
}
required_postprocessed_keys = {
    'discriminator_score',
    'fake_score',
    'real_score',
    'discriminator_probability',
    'fake_probability',
    'real_probability',
}
missing_signal_keys = sorted(required_signal_keys - set(check_batch.get('signals', {})))
missing_postprocessed_keys = sorted(required_postprocessed_keys - set(check_batch.get('postprocessed', {})))
assert not missing_signal_keys, f"Missing batch['signals'] keys: {missing_signal_keys}"
assert not missing_postprocessed_keys, f"Missing batch['postprocessed'] keys: {missing_postprocessed_keys}"

print('generated:', check_batch['signals']['generated'].shape)
print('discriminator logits:', check_batch['signals']['discriminator_logits'].shape)
print('fake logits:', check_batch['signals']['fake_logits'].shape)
print('real logits:', check_batch['signals']['real_logits'].shape)

answer['check_batch'] = check_batch
answer['generated_shape'] = list(check_batch['signals']['generated'].shape)
answer['discriminator_logits_shape'] = list(check_batch['signals']['discriminator_logits'].shape)
answer['fake_logits_shape'] = list(check_batch['signals']['fake_logits'].shape)
answer['real_logits_shape'] = list(check_batch['signals']['real_logits'].shape)
answer['bridge_class'] = gan_model.generator_discriminator_bridge.__class__.__name__


In [ ]:
course_utils.checker.print_progress(
    'Task 2: Build the Gradient-Reversal GAN', answer,
    prefixes=(
        'bridge_class',
        'check_batch',
        'generated_shape',
        'fake_logits_shape',
        'real_logits_shape',
        'discriminator_logits_shape',
    ),
    is_small_run=IS_UNITTEST)


## Model Graph

A model graph is a compact way to check the full adversarial signal path: noise becomes a generated image, generated and real images meet in the discriminator, and the classifier emits real/fake logits.

The gradient-reversal layer appears on the generated branch because it is the piece that turns one binary classification loss into opposite learning signals for the two networks.


In [ ]:
from torchview import draw_graph


class GANGraphAdapter(torch.nn.Module):
    def __init__(self, gan):
        super().__init__()
        self.gan = gan

    def forward(self, noise, real_image):
        generated = self.gan.generator(noise)
        reversed_generated = self.gan.generator_discriminator_bridge(generated)
        real_image = real_image.reshape(real_image.shape[0], -1)
        discriminator_input = torch.cat([reversed_generated, real_image], dim=0)
        features = self.gan.discriminator(discriminator_input)
        return self.gan.classifier(features).flatten()


gan_graph = draw_graph(
    GANGraphAdapter(gan_model),
    input_data=(torch.randn(4, noise_dim), torch.randn(4, 28, 28)),
    expand_nested=True,
    graph_name='GradientReversalGAN',
)
if not IS_UNITTEST and shutil.which("dot"):
    display(gan_graph.visual_graph)
elif not IS_UNITTEST:
    print("Install the Graphviz dot executable to display this graph.")

# Task 3: Create the Standard GAN Classification Loss, Optimizer, and Metrics

Create one binary classification loss over generated and real logits.

Implement `gan_classification_loss(batch)` with `torch.nn.functional.binary_cross_entropy_with_logits`:

- `fake_logits` should be compared with zero targets, because the discriminator should classify generated samples as fake;
- `real_logits` should be compared with one targets, because the discriminator should classify real MNIST samples as real;
- return `fake_loss + real_loss`.

Because generated samples pass through the gradient-reversal layer, this same loss trains the two sides in opposite directions. The discriminator is rewarded for recognizing fake samples, while the generator receives the reversed fake-sample gradient and is rewarded for making fake samples look real.

Implement `discriminator_accuracy(batch)` by thresholding logits at `0`:

- fake logits are correct when they are `< 0`;
- real logits are correct when they are `>= 0`;
- return `(correct_count, total_count)`.

Set:

- `losses = {'classification': gan_classification_loss}`;
- `metrics = {'discriminator_accuracy': discriminator_accuracy}`;
- one `AdamW` optimizer with two parameter groups: generator parameters at learning rate `2.0e-4`, discriminator plus classifier parameters at learning rate `1.0e-4`.

Do not add a learning-rate scheduler for this GAN task. GAN training is already a balance between two networks; changing both learning rates automatically from one validation loss can make that balance harder to reason about in this simplified setup.

The generator learning rate is intentionally a little higher than the discriminator learning rate. Together with the high-slope `LeakyReLU`, this helps the generator keep receiving useful updates when the discriminator learns the easy real/fake separation too quickly.


In [ ]:
# Edit this cell in task.ipynb.
def gan_classification_loss(batch):
    return batch['signals']['fake_logits'].abs().sum()


def discriminator_accuracy(batch):
    return 0, 1

losses = {'classification': gan_classification_loss}
metrics = {'discriminator_accuracy': discriminator_accuracy}
optimizer = torch.optim.SGD(gan_model.parameters(), lr=0.0)

## YOUR CODE HERE
# -- start --
def gan_classification_loss(batch):
    fake_logits = batch['signals']['fake_logits']
    real_logits = batch['signals']['real_logits']
    fake_targets = torch.zeros_like(fake_logits)
    real_targets = torch.ones_like(real_logits)
    fake_loss = F.binary_cross_entropy_with_logits(fake_logits, fake_targets)
    real_loss = F.binary_cross_entropy_with_logits(real_logits, real_targets)
    return fake_loss + real_loss


def discriminator_accuracy(batch):
    fake_logits = batch['signals']['fake_logits']
    real_logits = batch['signals']['real_logits']
    fake_correct = (fake_logits < 0).sum()
    real_correct = (real_logits >= 0).sum()
    correct = fake_correct + real_correct
    total = torch.tensor(fake_logits.numel() + real_logits.numel(), device=correct.device)
    return correct, total

losses = {'classification': gan_classification_loss}
metrics = {'discriminator_accuracy': discriminator_accuracy}
optimizer = torch.optim.AdamW(
    [
        {'params': gan_model.generator.parameters(), 'lr': 2.0e-4},
        {
            'params': list(gan_model.discriminator.parameters()) + list(gan_model.classifier.parameters()),
            'lr': 1.0e-4,
        },
    ]
)
# -- end --

loss_check_batch = {
    'data': {
        'noise': torch.randn(8, noise_dim),
        'image': torch.randn(8, 28, 28),
    }
}
gan_model(loss_check_batch)
test_loss_val = {name: fn(loss_check_batch) for name, fn in losses.items()}
test_metrics_val = {name: fn(loss_check_batch) for name, fn in metrics.items()}

answer['loss_names'] = list(losses.keys())
answer['metric_names'] = list(metrics.keys())
answer['optimizer_class'] = optimizer.__class__.__name__
answer['optimizer_defaults'] = dict(optimizer.defaults)
answer['optimizer_group_lrs'] = [group['lr'] for group in optimizer.param_groups]
answer['test_loss_val'] = test_loss_val
answer['test_metrics_val'] = test_metrics_val



In [ ]:
course_utils.checker.print_progress(
    'Task 3: Create the Standard GAN Classification Loss, Optimizer, and Metrics', answer,
    prefixes=(
        'loss_names',
        'metric_names',
        'optimizer_',
        'test_loss_val',
        'test_metrics_val',
    ),
    is_small_run=IS_UNITTEST)


# Task 4: Create DataLoaders

Create dataloaders for the noise-plus-MNIST datasets.

Use `torch.utils.data.DataLoader` with:

- `batch_size = 64`;
- `shuffle=True` and `drop_last=True` for `train_dl`;
- `shuffle=False` and `drop_last=True` for `valid_dl`.

The training loop expects each dataloader batch to contain `noise`, `image`, and `label`. The model will wrap each dataloader batch into `{'data': batch}` internally.


In [ ]:
# Edit this cell in task.ipynb.
batch_size = 16
train_dl = DataLoader(train_dataset, batch_size=batch_size, shuffle=False)
valid_dl = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False)

## YOUR CODE HERE
# -- start --
batch_size = 64
train_dl = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)
valid_dl = torch.utils.data.DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    drop_last=True,
)
# -- end --

train_batch = next(iter(train_dl))
valid_batch = next(iter(valid_dl))

pprint(train_batch)

answer['train_batch'] = str(train_batch)
answer['valid_batch'] = str(valid_batch)


In [ ]:
course_utils.checker.print_progress(
    'Task 4: Create DataLoaders', answer,
    prefixes=('train_batch', 'valid_batch'),
    is_small_run=IS_UNITTEST)


# Task 5: Train With the Shared Training Loop

Reuse `src.train_loop.train_model`. Do not write a custom GAN training loop, and do not pass a scheduler for this task.

For each batch, the `GAN.forward` method builds the combined fake-plus-real discriminator batch internally. The shared training loop computes the single classification loss, calls one backward pass, and lets the gradient-reversal layer send the opposite fake-sample gradient into the generator.

Store the returned histories for:

- training and validation classification loss;
- training and validation discriminator accuracy.


In [ ]:
n_epochs = 1 if IS_UNITTEST else FULL_RUN_EPOCHS

training_logger = src.utils.mlflow.prepare_mlflow_logger('gradient_reversal_gan')

training_history = src.train_loop.train_model(
    gan_model,
    n_epochs,
    train_dl,
    valid_dl,
    losses,
    optimizer,
    metrics=metrics,
    mlflow_logger=training_logger,
    run_name='gradient_reversal_gan',
)

train_gan_loss_history = training_history['train_loss']['classification']
valid_gan_loss_history = training_history['valid_loss']['classification']
train_discriminator_accuracy_history = training_history['train_metrics']['discriminator_accuracy']
valid_discriminator_accuracy_history = training_history['valid_metrics']['discriminator_accuracy']

training_logger.end()

answer['train_gan_loss_history'] = train_gan_loss_history
answer['valid_gan_loss_history'] = valid_gan_loss_history
answer['train_discriminator_accuracy_history'] = train_discriminator_accuracy_history
answer['valid_discriminator_accuracy_history'] = valid_discriminator_accuracy_history


In [ ]:
course_utils.checker.print_progress(
    'Task 5: Train With the Shared Training Loop', answer,
    prefixes=(
        'train_gan_loss_history',
        'valid_gan_loss_history',
        'train_discriminator_accuracy_history',
        'valid_discriminator_accuracy_history',
    ),
    is_small_run=IS_UNITTEST)


# Generated Images

After the short checker-friendly run, generated digits may still look rough. The goal here is not a polished generative model; the goal is to verify that the full adversarial path runs and produces image-shaped samples from noise.

The generator ends with `Tanh`, so plot generated images with the same `[-1, 1]` value range used for MNIST in the earlier tasks.


In [ ]:
with torch.no_grad():
    gan_model.eval()
    visualization_noise = valid_batch['noise'][:16]
    generated_images = gan_model.generator(visualization_noise).reshape(-1, 28, 28)
    


fig, axes = plt.subplots(2, 8, figsize=(10, 2.8), constrained_layout=True)
for index, ax in enumerate(axes.flat):
    ax.imshow(generated_images[index], cmap='gray', vmin=-1, vmax=1)
    ax.axis('off')
fig.suptitle('generated MNIST-like samples')
plt.show()

answer['generated_visualization_shape'] = list(generated_images.shape)
answer['generated_visualization_mean'] = float(generated_images.mean())

course_utils.checker.save_answer(answer)


In [ ]:
course_utils.checker.print_progress(
    'Generated Images', answer,
    prefixes=('generated_visualization_',),
    is_small_run=IS_UNITTEST)


In [ ]:
print(generated_images.min(), generated_images.max())


# Task 6: Experiment Time

The checker uses the short training run above. After that works, try longer experiments.

You can run experiments uncommenting the code below.

Although this code gives you some visible results, they are definitily
not good. Start with reducing learning rate to `2.0e-5` and `1.0e-5` for 
the generator and the discriminator respectively. 

Useful experiments:

- change the noise dimensionality;
- make the generator or discriminator wider;
- inspect how generated images change during training;
- try different `gradient_reversal_strength` values;
- compare smaller and larger `LeakyReLU` negative slopes. A very small slope can make the generator's gradients fade when the discriminator is already confident, while a larger slope keeps more signal moving through negative activations;
- manually tune the generator and discriminator learning rates. Avoid automatic schedulers at first, because a GAN's two-sided optimization can become harder to interpret when both learning rates change implicitly.


## Why GANs Are Hard To Train

The central difficulty in many GANs is not only that two networks are trained at once. A deeper problem is that the generator can receive very small or poorly shaped gradients when the discriminator works too well. If the discriminator confidently separates real and generated samples, its own classification signal is very informative, but the generator may get a weak training signal about how to move its samples toward the real data distribution.

Many GAN variants try to repair this gradient problem. Wasserstein GANs replace the classifier view with a smoother critic objective. MSE or least-squares GANs change the classification loss so gradients decay less abruptly. Other methods adjust architectures, regularization, update schedules, noise injection, normalization, or the way real and fake targets are represented.

These fixes help, but they do not remove the basic tension: the discriminator often gets a more direct and more informative gradient than the generator. Training a GAN is largely about keeping that game balanced enough for both networks to keep learning.


In [ ]:
# # # Optional long-run area. Do not include this in checker-facing answers.

# long_gan_model = src.models.feedforward.adversarial.GAN(
#     [256, 512, 28 * 28],
#     gradient_reversal_strength=1.0,
# )
# long_optimizer = torch.optim.AdamW(
#     [
#         {'params': long_gan_model.generator.parameters(), 'lr': 2.0e-5},
#         {
#             'params': list(long_gan_model.discriminator.parameters()) + list(long_gan_model.classifier.parameters()),
#             'lr': 1.0e-5,
#         },
#     ]
# )
# long_logger = src.utils.mlflow.prepare_mlflow_logger('gradient_reversal_gan_long_run')
# long_history = src.train_loop.train_model(
#     long_gan_model,
#     100,
#     train_dl,
#     valid_dl,
#     losses,
#     long_optimizer,
#     metrics=metrics,
#     mlflow_logger=long_logger,
#     run_name='gradient_reversal_gan_long_run',
# )
# long_logger.end()



In [ ]:
# with torch.no_grad():
#     gan_model.eval()
#     visualization_noise = valid_batch['noise'][:16]
#     generated_images = long_gan_model.generator(visualization_noise).reshape(-1, 28, 28)

# fig, axes = plt.subplots(2, 8, figsize=(10, 2.8), constrained_layout=True)
# for index, ax in enumerate(axes.flat):
#     ax.imshow(generated_images[index], cmap='gray', vmin=-1, vmax=1)
#     ax.axis('off')
# fig.suptitle('generated MNIST-like samples')
# plt.show()

# answer['generated_visualization_shape'] = list(generated_images.shape)
# answer['generated_visualization_mean'] = float(generated_images.mean())

# course_utils.checker.save_answer(answer)
